# Offline SageMaker local mode — scikit-learn pipeline

A single-`TrainingStep` SageMaker pipeline executed **locally** against moto via `sagemaker-local`. Pipeline steps run synchronously in local containers.

Dataset: `breast_cancer` (binary classification), loaded inside the container from scikit-learn.

In [ ]:
import os
from dataclasses import replace

from sagemaker.sklearn.estimator import SKLearn
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import TrainingStep
from sagemaker_local.config import config_from_env
from sagemaker_local.session import make_local_pipeline_session

cfg = replace(
    config_from_env(),
    bucket="sagemaker-scikit-learn",
    image_tag="sagemaker-scikit-learn:train",
)
boto_session, pipeline_session = make_local_pipeline_session(cfg)

## Define the pipeline

The estimator's `fit()` is captured as a `TrainingStep` request; nothing runs until the pipeline is started.

In [ ]:
est = SKLearn(
    entry_point="train.py",
    source_dir=os.path.join(
        os.environ.get("SAGEMAKER_LOCAL_REPO_PATH", "/workspace"),
        "projects",
        "sagemaker_scikit_learn",
        "src",
        "sagemaker_scikit_learn",
    ),
    role=cfg.role_arn,
    instance_type="local",
    instance_count=1,
    image_uri=cfg.image_tag,
    sagemaker_session=pipeline_session,
    output_path=f"s3://{cfg.bucket}/models",
    hyperparameters={"dataset": "breast_cancer"},
)
train_step = TrainingStep(name="train-breast-cancer", estimator=est)
pipeline = Pipeline(
    name="sklearn-local-pipeline",
    steps=[train_step],
    sagemaker_session=pipeline_session,
)

## Start it

`create()` registers the pipeline with the local client; `start()` executes every step synchronously in local containers.

In [ ]:
pipeline.create(role_arn=cfg.role_arn)
execution = pipeline.start()
print("pipeline started under name", pipeline.name)